# BazaarBot GRPO Training (Kaggle P100 / T4)

End-to-end `Run All` flow:

1. Install deps
2. HF auth via Kaggle Secret
3. Clone `bazaarbot_env` from GitHub
4. Load Llama-3.1-8B + LoRA (4-bit)
5. Build SFT dataset from a rule-based buyer
6. SFT warmup (teach the JSON format)
7. Post-SFT sanity check (verify valid JSON)
8. Build GRPO prompt dataset
9. Pre-GRPO diagnostic (verify reward variance)
10. GRPO training (with HF checkpoints)
11. Final push to HF

**Kaggle settings:**
- Accelerator: GPU T4 x2
- Internet: On
- Add-ons → Secrets → add `HF_TOKEN` (HF write token)

Cells are designed so the whole thing runs top-to-bottom without manual intervention.

## 1 · Install deps

In [ ]:
!pip install -q --upgrade "trl>=0.12" "peft>=0.13" "transformers>=4.46" "accelerate>=1.1" "bitsandbytes>=0.44" "datasets>=3.0" huggingface_hub

## 2 · HF auth

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("HF login OK")

## 3 · Clone bazaarbot_env

Idempotent: wipes any prior clone so `Run All` always uses the latest main.

In [ ]:
import shutil, sys, subprocess

REPO_URL   = "https://github.com/paymybills/BazaarBATNA.git"
LOCAL_PATH = "/kaggle/working/metathon"

if os.path.exists(LOCAL_PATH):
    shutil.rmtree(LOCAL_PATH)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, LOCAL_PATH], check=True)

if LOCAL_PATH not in sys.path:
    sys.path.insert(0, LOCAL_PATH)

# Force-reimport in case the notebook was already run once
for mod in list(sys.modules):
    if mod.startswith("bazaarbot_env"):
        del sys.modules[mod]

from bazaarbot_env import (
    BazaarGymEnv, format_observation, parse_action,
    DEFAULT_SYSTEM_PROMPT, strip_think_tags, TASKS,
)

print("bazaarbot_env loaded. Tasks:", list(TASKS.keys()))
assert "amazon_realistic" in TASKS, "stale clone — amazon_realistic missing"
# parse_action drops <think>...</think> blocks before JSON extraction
probe = parse_action('<think>x</think>{"action": "offer", "price": 5}')
assert probe.get("action") == "offer" and probe.get("price") == 5, f"parse_action broken: {probe}"
print("parse_action handles think-wrapped JSON: OK")

## 4 · Base model + LoRA

**Llama-3.1-8B-Instruct** — dense standard transformer, clean ChatML template
(no think tag shenanigans), mature TRL integration.  8B at 4-bit nf4 + LoRA
leaves plenty of T4 headroom for rollouts; quantizes to ~5GB for local Ollama.

If you hit a Meta license wall on HF, swap to `unsloth/Llama-3.1-8B-Instruct`
(same weights, ungated mirror).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "unsloth/Llama-3.1-8B-Instruct"
REPO_ID    = "PayMyBills/bestdealbot-v2"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

# prepare_model_for_kbit_training() enables gradient checkpointing, which
# silently corrupts generate() output on Llama (KV cache conflict with
# checkpointed forwards).  Our 3B + 4-bit + LoRA fits on T4 without it.
# SFTTrainer will re-enable it internally during its own training step.
model.gradient_checkpointing_disable()
model.config.use_cache = True

## 5 · Build SFT dataset

A cold 4-bit Llama can technically follow JSON instructions zero-shot but
reliability varies.  A quick SFT pass on 256 (prompt, rule-based-action) pairs
makes the model emit valid JSON ~100% of the time so GRPO's reward signal isn't
dominated by parse errors.

Llama 3.2's chat template is vanilla ChatML — no think tags, no template
quirks.  Assistant turn goes straight to content.

**v2 update:** SFT targets now include a `message` field rendered from
`nlp.templates`. This teaches the model to always speak natural Hinglish/English
alongside the structured action.


In [ ]:
import json
import random
import sys
from datasets import Dataset

# Templates module ships in the cloned repo. Add it to sys.path explicitly in case
# the cell-6 path insert was lost across kernel restarts.
if "/kaggle/working/metathon" not in sys.path:
    sys.path.insert(0, "/kaggle/working/metathon")
from nlp.templates import render as render_msg


def _rule_based_buyer(obs: dict) -> dict:
    """Heuristic baseline + templated message field.

    The message field is what makes the v2 model talk during inference. SFT
    teaches it that valid output is JSON with {action, price, message}, not just
    {action, price}.
    """
    ask    = obs.get("seller_asking_price") or obs.get("opponent_last_offer") or 100
    budget = obs.get("own_private_budget") or 100
    rnd    = obs.get("current_round") or 0
    last   = obs.get("own_last_offer")

    if ask <= budget * 0.5:
        return {"action": "accept", "price": None,
                "message": render_msg("accept", None)}
    if ask > budget:
        return {"action": "walk", "price": None,
                "message": render_msg("walk", None)}
    if rnd == 0 or last is None:
        price = ask * random.uniform(0.25, 0.40)
    else:
        price = last + (ask - last) * random.uniform(0.2, 0.35)
    price = max(1.0, min(price, budget * 0.8))
    price = round(price, 2)
    return {"action": "offer", "price": price,
            "message": render_msg("offer", price, ask=ask)}


def _build_sft_sample(task: str, seed: int) -> dict:
    env = BazaarGymEnv(task_name=task, seed=seed)
    obs, _ = env.reset()
    action = _rule_based_buyer(obs)
    full = tokenizer.apply_chat_template(
        [
            {"role": "system",    "content": DEFAULT_SYSTEM_PROMPT},
            {"role": "user",      "content": format_observation(obs)},
            {"role": "assistant", "content": json.dumps(action, ensure_ascii=False)},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": full}


SFT_N = 256
sft_rng = random.Random(1)
sft_rows = []
for _ in range(SFT_N):
    task = sft_rng.choice(["amazon_realistic", "single_deal", "asymmetric_pressure"])
    seed = sft_rng.randint(0, 1_000_000)
    sft_rows.append(_build_sft_sample(task, seed))

sft_ds = Dataset.from_list(sft_rows)
print(f"SFT dataset: {len(sft_ds)} examples")
print("Sample tail (verify {action, price, message} JSON):")
print(sft_ds[0]["text"][-250:])


## 6 · SFT warmup

~3–5 min on T4.  Loss should drop from ~1.8 to <0.3.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_cfg = SFTConfig(
    output_dir="/kaggle/working/bestdealbot-sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=1,
    logging_steps=5,
    save_strategy="no",
    bf16=True,
    report_to="none",
    max_length=1024,
    dataset_text_field="text",
    packing=False,
)

sft_trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_cfg,
    train_dataset=sft_ds,
)
sft_trainer.train()
print("SFT warmup complete.")

## 7 · Post-SFT sanity check

Generate 3 completions using the exact prompt format GRPO will use.

**Expected output:** valid JSON with `action`, `price`, and `message` fields, e.g.
```
{"action": "offer", "price": 21000, "message": "yaar 21000 mein de do bhai"}
```

The `message` field is what the v2 model outputs alongside the structured action — it's
populated from `nlp.templates` during SFT and is what makes the agent talk in the demo.
Llama-3.1 has no think tags; assistant turn goes straight to the JSON object.


In [ ]:
# SFTTrainer re-enables gradient checkpointing during training.
# generate() needs checkpointing OFF and use_cache ON, otherwise the KV cache
# silently corrupts and the model emits garbage tokens.
model.gradient_checkpointing_disable()
model.config.use_cache = True

GEN_MAX_NEW_TOKENS = 128  # was 48 — bumped for the {action, price, message} schema

@torch.no_grad()
def _generate(prompt_text: str, max_new_tokens: int = GEN_MAX_NEW_TOKENS) -> str:
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


env = BazaarGymEnv(task_name="amazon_realistic", seed=7)
obs, _ = env.reset()
_sanity_prompt = tokenizer.apply_chat_template(
    [{"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
     {"role": "user",   "content": format_observation(obs)}],
    tokenize=False, add_generation_prompt=True,
)

print("Prompt tail:", repr(_sanity_prompt[-80:]))
print()
any_parsed = False
for i in range(3):
    out = _generate(_sanity_prompt)
    parsed = parse_action(out, fallback_price=0)
    ok = not parsed.get("_parse_error")
    any_parsed = any_parsed or ok
    print(f"sample {i}: {out!r}")
    print(f"   parsed: {parsed}  {'OK' if ok else 'PARSE_ERR'}")

assert any_parsed, "SFT did not teach JSON format — at least one sample must parse"

## 8 · Build GRPO prompt dataset

A GRPO `prompt` = (task, seed) encoded as the tokenized chat up to the assistant
marker.  The reward function will re-sample the env from `seed` and `task` so
each completion is scored against the same scenario (required for group-
normalized advantages).

In [ ]:
TRAIN_TASKS = ["amazon_realistic", "amazon_realistic", "single_deal"]
N_PROMPTS = 256

grpo_rng = random.Random(0)
train_rows = []
for i in range(N_PROMPTS):
    task = grpo_rng.choice(TRAIN_TASKS)
    seed = grpo_rng.randint(0, 1_000_000)
    env = BazaarGymEnv(task_name=task, seed=seed)
    obs, _ = env.reset()
    chat = tokenizer.apply_chat_template(
        [{"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
         {"role": "user",   "content": format_observation(obs)}],
        tokenize=False,
        add_generation_prompt=True,
    )
    train_rows.append({"prompt": chat, "task": task, "seed": seed})

train_ds = Dataset.from_list(train_rows)
print("Training prompts:", len(train_ds))
print("Task mix:", dict((t, sum(1 for r in train_rows if r['task'] == t)) for t in set(TRAIN_TASKS)))
print("Prompt tail:", repr(train_rows[0]["prompt"][-80:]))

## 9 · Reward function

FAST_MODE: no continuation rollouts, shaped first-step reward.  The shape
curve is a bell around 25% of the seller's ask — lower is better up to a
point, then diminishing returns, then collapse (aggressive offers cause
seller walk).  Continuous so sampled completions with different prices
produce different rewards → nonzero GRPO advantages.

In [ ]:
import math

FAST_MODE = True
MAX_CONTINUATION_TURNS = 0 if FAST_MODE else 6

# After SFTTrainer finishes, gradient checkpointing is re-enabled on the
# model (and use_cache disabled).  Flip both back for generate() — GRPO's
# training_step will re-enable checkpointing when it needs it.
model.gradient_checkpointing_disable()
model.config.use_cache = True


def _shaped_first_step_reward(obs, action, step_reward):
    if action.get("_parse_error"):    return -0.3
    act = action.get("action")
    ask = obs.get("seller_asking_price") or 0
    budget = obs.get("own_private_budget") or 0
    price = action.get("price") or 0

    if act == "accept":               return -0.2
    if act == "walk":                 return -0.3
    if ask <= 0 or budget <= 0:       return float(step_reward)
    if price <= 0 or price > budget:  return -0.3

    # Bell curve peaking at ratio=0.25, decaying ~sigma=0.28
    ratio = price / ask
    shape = math.exp(-((ratio - 0.25) ** 2) / 0.08)
    return float(step_reward) + shape


def reward_fn(completions, prompts=None, completion_ids=None, **kwargs):
    rewards = []
    tasks = kwargs.get("task") or ["amazon_realistic"] * len(completions)
    seeds = kwargs.get("seed") or [None] * len(completions)

    for completion, task, seed in zip(completions, tasks, seeds):
        env = BazaarGymEnv(task_name=task, seed=seed)
        obs, _ = env.reset()
        first_action = parse_action(completion, fallback_price=obs.get("own_private_budget", 100) * 0.3)
        _obs, r, done, info = env.step(first_action)

        if MAX_CONTINUATION_TURNS > 0:
            turns = MAX_CONTINUATION_TURNS
            while not done and turns > 0:
                chat = tokenizer.apply_chat_template(
                    [{"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
                     {"role": "user",   "content": format_observation(_obs)}],
                    tokenize=False, add_generation_prompt=True)
                act = parse_action(_generate(chat), fallback_price=_obs.get("own_private_budget", 100) * 0.3)
                _obs, r, done, info = env.step(act)
                turns -= 1
            rewards.append(float(env.score()))
        else:
            rewards.append(_shaped_first_step_reward(obs, first_action, r))
    return rewards

## 10 · Pre-GRPO diagnostic

End-to-end check before burning GPU time.  Verifies completions vary AND rewards
vary.  Raises AssertionError if either check fails — better to fail here than
watch 30 steps of zero loss.

In [ ]:
sample = train_ds[0]
prompt = sample["prompt"]
task   = sample["task"]
seed   = sample["seed"]

print(f"Task: {task}  Seed: {seed}")
print(f"Prompt tail: {prompt[-100:]!r}")
print()

completions = [_generate(prompt) for _ in range(4)]
for i, c in enumerate(completions):
    print(f"[{i}] {c!r}")
print()

parsed = [parse_action(c, fallback_price=0) for c in completions]
for i, p in enumerate(parsed):
    flag = " (PARSE ERR)" if p.get("_parse_error") else ""
    print(f"[{i}] action={p.get('action')} price={p.get('price')}{flag}")
print()

rewards = reward_fn(completions, task=[task]*4, seed=[seed]*4)
mean = sum(rewards) / 4
std  = (sum((r - mean) ** 2 for r in rewards) / 4) ** 0.5
print(f"Rewards: {[round(r, 3) for r in rewards]}")
print(f"Mean: {mean:.3f}  Std: {std:.4f}")

parse_errs = sum(1 for p in parsed if p.get("_parse_error"))
assert parse_errs <= 1, f"{parse_errs}/4 parse errors — SFT didn't take"
assert std > 0.01, f"reward std too low ({std:.4f}) — GRPO will produce zero advantages"
print("\nDIAGNOSTIC PASSED — ready for GRPO")

## 11 · GRPO training

`max_steps=30` for the smoke run.  Bump to 200+ once you've confirmed the
reward curve trends upward.  HF checkpoint push every 10 steps protects
against Kaggle session timeouts.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_cfg = GRPOConfig(
    output_dir="/kaggle/working/bestdealbot-ckpt",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=2,
    max_completion_length=48,
    learning_rate=5e-6,
    num_train_epochs=1,
    max_steps=30,
    logging_steps=1,
    save_steps=10,
    save_total_limit=2,
    bf16=True,
    report_to="none",
    remove_unused_columns=False,
    push_to_hub=True,
    hub_model_id=REPO_ID,
    hub_strategy="checkpoint",
    hub_private_repo=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=grpo_cfg,
    train_dataset=train_ds,
)
trainer.train()

## 12 · Final push

Adapters only (~50MB).  Merge base+adapter locally after download.

In [ ]:
model.save_pretrained("/kaggle/working/bestdealbot-adapter")
tokenizer.save_pretrained("/kaggle/working/bestdealbot-adapter")

from huggingface_hub import HfApi
HfApi().upload_folder(
    folder_path="/kaggle/working/bestdealbot-adapter",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="GRPO adapter final from Kaggle run",
)
print(f"Pushed to https://huggingface.co/{REPO_ID}")

## Local post-training

```bash
hf download PayMyBills/bestdealbot --local-dir ~/models/bdb

python -m peft merge_and_unload \
    --base-model unsloth/Llama-3.2-3B-Instruct \
    --adapter ~/models/bdb \
    --out ~/models/bdb-merged

python llama.cpp/convert_hf_to_gguf.py ~/models/bdb-merged \
    --outfile bdb.gguf --outtype q4_k_m

cat > Modelfile <<'EOF'
FROM ./bdb.gguf
SYSTEM """You are a skilled buyer negotiating at an Indian bazaar."""
EOF
ollama create bestdealbot -f Modelfile
ollama run bestdealbot
```